## CASH Notebook

## Celestial Chase - LVL 1: The Teal Beacon

You've just woken up from cryo-sleep. No memory. No crew. Just you, a dying sun, and a faint signal pulsing from the sky.

One star glows different from the rest. Not white. Not grey. **Teal.**

Find it. Its position holds part of the key. But position alone is not enough - you must also know how much of its face is lit by the sun.

---

**The encoded signal:** `uretpjy`

**Your task:**
1. Display the image and find the **cyan pixel** - it is the only pixel where `B == 255` and `G == 255` and `R == 0`
2. Read its `(x, y)` coordinates
3. Use `ephem` to compute **Venus's phase** (`int(venus.phase)`) on `2025/6/21 12:00:00` UTC from Zurich (lat=`47.3769`, lon=`8.5417`)
4. Build the key: `key = x * y + int(venus.phase)`
5. Build the permutation: `random.Random(key).shuffle(list(range(len(encoded))))`
6. Reverse the transposition to decode: `decoded[i] = encoded[perm[i]]`


In [ ]:
import base64, cv2, numpy as np
from IPython.display import display, Image as IPImage

# Starfield image embedded directly in this notebook
img_b64 = "iVBORw0KGgoAAAANSUhEUgAAAZAAAAGQCAIAAAAP3aGbAAASF0lEQVR4Ae3BB3jXhZ3H8c/3eu1dH59OK1Wpigu1qGjFgYKyZIcVRhhhSkBCJAIBJIwwgkAAgyEIQWYYIRBW2CKgqDiwToriQi1qsXY+Xu/au8s9D/f4PPgINeM/ft//7/16mULj9Y9evvHSWwTALRMABN6Rdw83uKqhCQCcMCHeeqd3X1NYKgDfxgQATpgAwAkTADhhqoTsvDG5WbMFAHFlAhAkH//1/Ut+cIVwNiYgqBaunT+s1wgBXzEBgBMmAHDCFFf7juxq0aCNAKASTIiE1z48Uv+yBsK3ycxJz88pFFAtJgBwwgQEwMZ967q26CnUwBsf//qGS36lhGYCACdMAKIsO29MbtZsocZMQBS89dkb1154gxLa/JVzRvQbLcSQCWdz6u8na32vtgAEiQnnljZmQNHs5UJVDJ8wZMH0xQKiwAQA0bdw7fxhvUaoZkxAVQwa1Xfp3FUC4sEEABEyf+WcEf1G6+uOnzpat1Y9RYKpug6++kSTm+4V4E1Sauvy4t2CQyYAOJvxs7NmjMlTkJgAwAkTADhhAgAnTKFXVFqY1j1diKsDr+xtenNLRdrxU0fr1qonRFOn/u22rNihmDABgBMmAHDCBABOmIDAW7Jh4eBuwxQmu58vb31HkmJryNiBi2ctU8Cc/PJE7fPq6DQTADhhAgAnTDWwqKRgaEqGACAmTAAQbyf+9E6dH1+tb2PCGTImDi2YtkgAAskEAE6YgKro0LfNtlW7BMSDCQCcMAGAEybAmx5DuqxfvEkIH1MA7H1xR8vb2gk18O7vj131s+sEJDQTADhhAgAnTABcWb9ndY9WfRRKJgBwwlQ5zx49eFe9JgJwhtlFuWPSsoVoWrdrVc82fXWaCQCcMAGAEyacISm1dXnxblVa2ZMlyc1TBCAmTLGy5eCGTk26CQCqywQAZ1Ncviw1aaCCxORfy+5N95YeEHA2b/72let/cbOQEExfN3HuuGmjZgpRNn521owxeUKNvf27N6/5+fVCOJgqZ8jYgYtnLRMAxI8JAJwwAee249kt7e7qJJzDuJmjZo6bK8SK6Rza9Gyxa90+odJadL1n38anhER3b7cmT2w4KMSDCQCcMAGAEyYA35AxcWjBtEVCwJgAwAkTADhhCrCS3cUprVOFmJtWMGlixlSFQGpGSnFBieCECUAVPfPmgUbXNxVizhRzX/zPZ+d/50IBQBWZAMAJEwAE1e/+67c//7df6CsmIPBe+/BI/csaCKFnAuJk+zOb2zfqLKDSTADghAkAnDABgBMmAHDCBCSuXYe3tWnYQUgUJgBwwuTQkXcPN7iqoQCEjAkAnDABgBMmxET3tM6lRZsFoAZMMbTj2S3t7uokAKgWE4Boev8Pb1/x02uESDABgBMmAHDCBMCVTfvXd2nWQ6FkAkKmVY9me9bvF6SC4nkZqSPlhymxzF85Z0S/0QKQiEzStqfLOtydLAAINhMAOGECgHPIzhuTmzVbgWECKiFtzICi2csFxJUJgEPHTx2tW6ueQsYEAE6YTvvNJ6/98uL6AhBI81fOGdFvtELPBABOmFADH//1/Ut+cIUAxIQJAJwwAYATJgBwwgQATpgAwAkTgEg79unr1110oxBpJgBwwgQE2zNvHmh0fVMBkgkAnDABgBMmAHDClHBShiaXLCoTgEpr0fWefRufUuCZAMAJEwA4YUL4bHu6rMPdyQK8MQHerNmxone7/kL4mADACVMgvXj82dvq3iUAOIMJCW3momnjhk4UkBBMSDiDs/ovyVshIOGYAMAJE86mf2bvFflrBA+KSgvTuqcLIWACACdMAOCECUDNVVTITIgyE4AaqqjQ/zMToskEoOYqKmQmRJkJAJwwAYATJgCx0qpHsz3r9wvVZQJCY88L21vd3l5wywSguopKC9O6pwuxYoIfT7227576LQSElQkAnDDFT96SGVmDxwsAKscEAE6Yqm755qIBndMEhEDrlOa7S54UgsEEAE6YAMAJE4DQyMxJz88plFsmRMjO57a2vbOjpNzCnOz0HCHA0rPTCnOLBG9MsdKxX9utK3cKcXLojf2Nb2gmwDMTgHN7cMrwRyYvUDSVH9qU1LiLUAkm+PT7//70Z/96kYAwMQGAEyYAcMIEAE6YIqrLwKRNy8oFAFFg8mz+yjkj+o0WgHAwAYATJkTfEy/tvPfWtvKp/NCmpMZdlKD2vLC91e3tBSdMSGiPb3zsvq73C0gIJsRDTn52TmauAFSFCQCcMAFAFKzfs7pHqz6KKBPCbdj4wQtnLBFqbNW2pX07DBKiyQQATphOmzBn7PTRs+TfA5Puf3TqYwKi7NFVcx/oO0qILRMAOGGKqE79221ZsUMAEAUmAHDCFFHJgzqULd0mAIgCE0Lv+KmjdWvVk0OzFk8fO2SCKuHklydqn1dHX+kyMGnTsnKFzIjJw+ZPWSjPTADghAkAnDABMVFcviw1aaCAGjABQDxsPlDauWl3VYUJQCCNmvbA3ImPCmcw4dv8+v0XfnXF7QIQbyaHXn7v+VuuvEMAgmTDE2u73dtL0WRKIFm5mXnZ+QKQoEwA4IQJCLfDx55ueN3dipPXPjxS/7IGQuWYAMAJEwA4YYqHNj1b7Fq3TwBQFSYAcMIEAE6YAMAJEzxLGZpcsqhMQDiYoixlaHLJojIF1bSCSRMzpgqIvuRBHcqWbhNqwAQATpiib9P+9V2a9VBcff6PTy747sUC4JkpHj7/xycXfPdiodJ63t913WMbBYSbCc71vL/rusc2CggB0ze89M5zt159pwCE0tKyRYOShyqQTEBYdRmYtGlZueCHCQCcMAE4t4cfm/rQ/ZOEYDA5tHLr4/063icAIWMKmE//9tFF379UAPANJgDV9eqJl26qc6sQKyZAOvLu4QZXNRQQbCYAcMIEJJy+D/Rc9eg6IeGYAMAJEwA4YQIAJ0wAnPhRRcWfzRRiJgBwwgQATpgAJLQP/nj88p/UVUIwAYATJqAGlpYtGpQ8VEgIr314pP5lDRRgpoh6/q1Dd1zbWECUvXj82dvq3iWEjAkIhqMnX61X+yYB52YCACdMAOCECQCcMNXYsPGDF85YIgCIMhMAOGHCOXQe0H7z8u0CEBgmAHDCBCBRFJUWpnVPVxX9+v0XfnXF7fLAhAhZs2NF73b9BSBqTADghAkJrezJkuTmKQISggkAnDABqKKxD4+c9dA8IeZMcKugeF5G6kgBoWECKmHQqL5L565SuP2wouIvZoqC537z1J2/vEfVteeF7a1ub68QMAGAEyYAcMKE4Hnlgxdvvvw2Afg6E77h/T+8fcVPr9E3ZEwcWjBtkSLtXyoq/tdM8fP279685ufXC1H2ygcv3nz5bYqfhx+b+tD9k+SZCQCcMAGInAenDH9k8gIhOkwA4ITp3B6aNfrhsXMEAMFgAgAnTMDZdB7QfvPy7QKCxHQ2T7/+5N03NhfgXK9h3dYu3CAkChMAJ5JSW5cX71aImQDACZO05eCGTk26CQCCzRQTXQYmbVpWLgCoARMAOGFC4C1Y/cjwPg8KCD0TADhhAgAnTADghCn62vRssWvdPgFAzZjOkJOfnZOZK1RLakZKcUGJgGjqnta5tGizwqF075ruLXvrDCYAcMIEAE6Y3Op6X8eNj28VgNAwAYATJgBwwgQATpgAwAkTADhhAuJnyNiBi2ctUyi9euKlm+rcKlSFCdHRbXCnDUu2CEDkmADACVMievr1J+++sbni5OSXJ2qfV0fhs2D1I8P7PCggakwIknEzR80cN1fAaSOnZsybVCB8xQTEycipGfMmFQhVdOiN/Y1vaKZQMgGAEyYACLAVW5b07zRYp5kAaffz5a3vSBKi6ci7hxtc1VCoARMQYpsPlHZu2l1wwoRoatvr3p1rn1B1nfjTO3V+fLVibsvBDZ2adFO4Fa7JT++dKQSJCUAVlT1Zktw8RYg5ExJCVm5mXna+gIRmAgAnTADghAmIocyc9PycQgHVYgIAJ0zndvLLE7XPqyMACAYTqiV/RV5m/ywBiCETADhhAgAnTEhQ733x1pXnXysggZgSVMrQ5JJFZQKQQExIIJ0HtN+8fLuABGUCACdMAZY8qEPZ0m1CpKVmpBQXlAiorpVbH+/X8T7FnAmV88ybBxpd31SA1L5Pq+2r9wgxZ0KCSs9OK8wtEpBATFE2f+WcEf1GCwBOGzdz1Mxxc1UtJgBwwhR9Lbs33Vt6QEBC63pfx42PbxWiyYRIe/m952+58g4BiDQTECRdBiZtWlYu4GxMQKT1Gd5j9YL1AiLNBCSEgSNTl80rFhKaCQCcMMXKZ//58YX/fokQPB37td26cqdQXYVr8tN7Zwrn8MX/fHb+dy5UJJgQJn2G91i9YL0An0wA4IQpBI6fOlq3Vj0BnvXP7L0if40qYdbi6WOHTFAiMsGnHkO6rF+8SUCYmGomMyc9P6dQIfCLiorfmgmuPLbu0ft7PiAkChMAOGECACdMAOCECQCcMAEBk7dkRtbg8QK+wQQATphCoHTvmu4tewtft+vwtjYNOwjww4RzeOmd5269+k4BCAwTEH35K/Iy+2cJYTJ9weQJw6eoul5657lbr75TX2cCACdMwBm63tdx4+NbBQSSCQCcMAGAE6YgycnPzsnMVVztfG5r2zs7CkAlDM7qvyRvhWLFFB3jZ2fNGJMnV9buXNmrbT8BCCoTEG9LyxYNSh4qhNvu58tb35Gkf8oEAE6YAMAJE7wZN3PUzHFzFUMf/eW9S394pRAm7/7+2FU/u04BYwKCpNewbmsXbhBwNiYAqK7pCyZPGD5FsWICAmnY+MELZywRcAYTADhhApwYNn7wwhlLFGBJqa3Li3erWtbtWtWzTV9F2rxls0YOHKtEYQIAJ0wAEtrn//jkgu9erIRgOptTfz9Z63u1BcRJenZaYW6RgK8zJbTsvDG5WbMFICGY4NZ7X7x15fnXSho9fcScCfMFJDoTgNhKGzOgaPZyoepMiKEuA5M2LSsXEt0n//HhsWPHmt/SWogoE5ybtXj62CETBISACQCcMAEIk+kLJk8YPkU+mQBIjyyf/eCAMcI3bH9mc/tGnRVRB199oslN96rqTBG1dufKXm37CQCiwAS3jn36+nUX3SggNEyIoW1Pl3W4O1kAqsUEAE6YgNAr2V2c0jpVCDwTPJixcMr4YZMFhJsJAJwwBc+rJ166qc6t+jan/n6y1vdqC0BomODBjyoq/mymAGvXu+WONXsFRJMJAJwwAYATJgDxlpWbmZedr5hLzUgpLihRRI19eOSsh+YpOkwA4ISt3bmyV9t+AuDE9AWTJwyfIieOnny1Xu2bFCEmAHDChH/qhbefuf2aRgIQACYAcMIExErfB3quenSdUEXvfP6bqy/4pSCZouD5tw7dcW1jBVj7Pq22r94joFq6DEzatKxcAfD6Ry/feOktCg0TEEgZE4cWTFsk4AwmAHDCBADRNH521owxeYoEEwA4YQIAJ0zfpn2fVttX7xEAxJsJQBV16t9uy4odQsyZgDN0T+tcWrRZQCCZAMAJExAwR9493OCqhoL01Gv77qnfQviKCUBCy84bk5s1WwnBBABOmADACRMAOGFCmIyYPGz+lIUCfDIhhkZNe2DuxEcVQ5/+7aOLvn+pgIRgAgAnTADghAk4w87ntra9s6Mi5PCxpxted7fgyootS/p3GqxAMgXe9mc2t2/UWQBCpnVK890lT+oMpgDY+tTGjvd0FYLq3m5NnthwUEgI0xdMnjB8inwyAUA0ffDH45f/pK4iwQQATpiA6uo2uNOGJVsExIopxHLys3Myc4XTTvzpnTo/vloIvQWrHxne50EFkinRle5d071lbwHwzwSgBgqK52WkjhRiwhQTJ788Ufu8OgKAGjABgBMmAHDCBABOmAB83cZ967q26CkEjwkAnDABCefQG/sb39BMCJ5O/dttWbFD1WUCACdMAOCEKcqePXrwrnpNBAA1ZgIAJ0xA/GTmpOfnFAphcuiN/Y1vaKZqMQGAEyYAcMIEAE6YnEvPTivMLRKAEDBFWVZuZl52vgCgxkxAIBWVFtavX//2axoJ+IoJAJwwAaiu5EEdypZuE2LFBATefaP7PT5npRB6JgBwwoQQSM9OK8wtEuCcCQCcMAFA5Hz6t48u+v6lioQVW5b07zRYZzABgBMmAAi8SfMemjry4f8D7BEwzbd4QwAAAAAASUVORK5CYII="

img_bytes = base64.b64decode(img_b64)
img_arr   = np.frombuffer(img_bytes, dtype=np.uint8)
img       = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)

cv2.imwrite('starfield.png', img)
display(IPImage('starfield.png'))
print("Image shape:", img.shape)


In [ ]:
import ephem, random

encoded  = "uretpjy"
obs_date = "2025/6/21 12:00:00"
obs_lat  = "47.3769"
obs_lon  = "8.5417"

# TODO Step 1: Find the cyan pixel (B=255, G=255, R=0) in img
# Hint: use np.where or a loop
pixel_x, pixel_y = 0, 0  # replace with actual coordinates

# TODO Step 2: Compute Venus phase with ephem
obs = ephem.Observer()
obs.lat  = obs_lat
obs.lon  = obs_lon
obs.date = obs_date
venus = ephem.Venus()
venus.compute(obs)
phase = 0  # replace: int(venus.phase)

# TODO Step 3: Build key and permutation
key  = pixel_x * pixel_y + phase
perm = list(range(len(encoded)))
random.Random(key).shuffle(perm)

# TODO Step 4: Reverse the transposition
# Hint: if encoded[perm[i]] = original[i], then decoded[i] = encoded[perm[i]]? Think carefully.
def transpose_decode(encoded, perm):
    pass  # implement this

answer = transpose_decode(encoded, perm)
print(answer)
